# 5.8 Loss 下降代表任務做得更好嗎？


希望模型寫出 [1,2,3,4]。在正確前文下，各位置預測 [1,2,0,4]，有三個位置對；自由生成卻接成 [1,2,0,0]，只剩兩個位置對。兩份都沒有交出完整答案，所以逐位置的表現與整題是否完成要分開記。

訓練時每個位置都拿到資料中的正確前文（標準序列），這叫 teacher forcing；自由生成則把自己剛選的字接回去當前文，第三個位置錯成 0，就改變了第四個位置看到的條件。下面先用這兩份手寫的預測算分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
truth = [1, 2, 3, 4]
teacher_forced = [1, 2, 0, 4]
generated = [1, 2, 0, 0]
for name, prediction in [("正確前文下", teacher_forced), ("自由生成", generated)]:
    correct = sum(a == b for a, b in zip(truth, prediction, strict=True))
    token_accuracy = correct / len(truth)
    exact_match = prediction == truth
    print(name, "逐token正確率", token_accuracy, "整串相同", exact_match)

`truth` 是正確答案（ground truth），兩份 `prediction` 分別模擬正確前文下與自己前文下的結果。`zip` 把同位置的兩項配成一對（[1.3](https://birdhackor.github.io/tiny-perceptron-vlm/1.3.html) 用過）；`strict=True` 要求兩份清單長度相同，否則提出 ValueError 並停止，不會默默略過較長清單的尾端。`a == b` 成立算 1、不成立算 0，`sum` 數出答對的位置數；除以答案長度 4，就得到 **token accuracy**（逐 token 正確率）。token 是模型一次讀寫的一個文字單位，本例一個字就是一個 token；tokenizer 怎麼把文字切成 token，[第 6 章](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html)才正式說明。第一份三個位置正確，輸出 0.75；第二份兩個位置正確，輸出 0.5。

`prediction == truth` 比較整份清單，順序、長度與每個位置都相同才是 True，這叫 **sequence exact match**（整串完全相同）。兩份都是 False。像算術答案、固定標籤這類任務，整串相同常比逐 token 的比例直接；開放式的表達則有許多合理寫法，完全相同不一定反映好壞，評分方法要依任務選擇。

代價量的是正確答案得到多少機率，不只量最高分的候選有沒有選對：正確字的機率是 0.6 或 0.9 都可能排第一，代價卻不同（[1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)）。所以代價下降，不保證模型接著自己寫時寫得出好句子。例如補充區那個用英文短篇故事訓練的模型，驗證代價已明顯下降；給它開頭 `Once upon a time there w`、每次選最高分的候選，它接出 `as a big was a she was a ber the`：用上了常見的英文片段，卻還沒組成清楚的句子。代價與實際生成的回答要一起看。

還要確認題目提供了答案需要的線索。例如想重現一段包含左右位置的原文，提示卻只給顏色和形狀、沒給左右時，沒重現原文不能直接算成答錯已知事實。

練習只把 `teacher_forced` 第三個位置的 0 改成 3。先預測第一份逐 token 正確率變 1、整串相同變 True，第二份仍是 0.5 和 False，再執行核對。這次只改了示意資料，沒有真的改變生成模型；要提升第二份的表現，還得在實際生成中驗證訓練是否有效。

<details>
<summary>補充：實驗紀錄與重做</summary>

再看兩個真正訓練後的例子。[5.1](https://birdhackor.github.io/tiny-perceptron-vlm/5.1.html) 補充的屬性短文模型，驗證資料的平均代價從 5.73454 降到 0.96282；可是給它開頭 `color=blue;shape=circle;`、每次選最高分的候選時，它接出 `side=right.`，那篇驗證原文剩下的卻是 `side=left.`。代價改善了，並沒有讓自由生成完整重現這篇原文。

這裡還有一個重要的判斷：提示只給顏色與形狀，沒有告訴模型物體在哪一邊；左或右都可能是格式合法的續寫。所以這個差異說明的是「沒有重現驗證原文」，不能單憑它判定右邊是錯誤的事實。若任務真的要判斷左右，就要在輸入提供相應的線索，另定答案與成功條件。文字接續的代價、寫出合法格式、回答有正確答案的問題，各自需要相應的評估，這也是後面另做[屬性問答](https://birdhackor.github.io/tiny-perceptron-vlm/7.12.html)的理由。

換成真正的故事，這個差別更容易看到。課程固定實驗 `real_text` 用 409 篇 TinyStories 英文短篇故事訓練，在另外 51 篇驗證故事的全部 42,453 個有效目標上，平均代價從 5.76091 降至 1.80083。但是讓它每次選最高分的候選、最多新增 32 個 token，一筆實際結果如下。這個模型把文字拆成 byte（位元組），本例的英文字母與空格各占一個 byte；表示寫完的結束標記另占一個 token，第 6 章起寫作 EOS；切詞方法在 [6.1](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html) 展開。

```text
給模型的開頭：Once upon a time there w
新生成的部分：as a big was a she was a ber the
```

它開始接出常見的英文片段，卻還沒有把片段組成清楚的句子，更沒有講出一個故事。這段輸出只是一筆觀察；平均代價用的是全部驗證文字，兩者不能互相取代。[實驗紀錄（JSON）](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/real_text.json)保留完整結果：`results.runs.tinystories.after.validation` 的 `nll` 是驗證代價、`effective_tokens` 是有效目標數，`samples` 是前 8 篇驗證故事的生成示例；前 8 篇測試故事的示例另外放在 `results.runs.tinystories.after.test.samples`。兩份都照實保留，沒有因為續寫不好就省略。這也給我們一個實用的檢查順序：先確認模型確實學到一些文字規律，再問它能否在自己的前文下完成想要的任務。重做這組實驗的指令在 [T.4](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4) 的補充區：先用 `scripts/fetch_training_assets.py --asset tinystories --asset chinese-poetry` 取得故事與詩文資料包，再跑 `--experiment real_text`。

</details>
